# 03 — Manifest → priors → scale-conditioned mass regressor (GPU job 2)

The novel model: mass regression conditioned on measured metric geometry (docs/MODELS.md §4). Requires **notebook 01** (Nutrition5k on Drive).

Steps: (1) extract the geometry manifest from the RGB-D captures — the same quantities the phone measures; (2) fit the κ/φ/h̄ shape priors used by the pure-geometry pipeline; (3) train the regressor.

- **Runtime:** manifest extraction ~30–60 min (CPU-bound); training ~1–2 h on H100 at batch 128.
- **Benchmarks to beat:** 26.1% calorie MAPE (RGB) / 16.5% (RGB+depth).
- All artifacts and caches persist to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/physics-powered-portion-estimation'

# Pretrained-backbone caches persist on Drive too.
os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf-cache'
os.environ['TORCH_HOME'] = f'{DRIVE_ROOT}/torch-cache'

DATA = f'{DRIVE_ROOT}/data/nutrition5k'
OUT = f'{DRIVE_ROOT}/out'
CKPT = f'{DRIVE_ROOT}/checkpoints/mass-regressor.pt'
!mkdir -p "{OUT}"
!nvidia-smi | head -12

In [ ]:
!git clone --depth 1 https://github.com/Hakeem-Hannoon/physics-powered-portion-estimation.git /content/ppe 2>/dev/null || (cd /content/ppe && git pull)
%pip -q install timm pandas

In [ ]:
%cd /content/ppe
# 1. Geometry manifest (area, height, volume per dish + mass/kcal labels).
!python model/data/prepare_nutrition5k.py --root "{DATA}" --out "{OUT}/n5k-manifest.csv"
!head -3 "{OUT}/n5k-manifest.csv"

In [ ]:
# 2. Shape priors — the printed global κ replaces DEFAULT_KAPPA in
#    packages/pipeline/src/estimate.ts and feeds nutrition/'s shape_priors.
!python model/priors/fit_priors.py --manifest "{OUT}/n5k-manifest.csv" --out "{OUT}/priors.json"
!cat "{OUT}/priors.json"

In [ ]:
# 3. Train. Per-epoch mass/kcal MAPE prints; best checkpoint saves to Drive.
!python model/train/mass_regressor_nutrition5k.py \
  --manifest "{OUT}/n5k-manifest.csv" \
  --epochs 50 --batch-size 128 \
  --output "{CKPT}"
print('README row template:')
print('| Nutrition5k test split | calorie MAPE | 26.1% RGB / 16.5% depth | **<best kcal MAPE from above>** |')